# 🛒 Modelo Predictivo de Intención de Compra en E-commerce

**Materia:** Herramientas para la Toma de Decisiones Basadas en Datos
**Algoritmo:** XGBoost Classifier
**Dataset:** Eventos de e-commerce estratificados (~1M de registros)
**Objetivo:** Predecir si un evento de navegación termina en compra (`is_purchased`)

---

### 📋 Estructura del Notebook
1. **Configuración del entorno** — Instalación e imports
2. **Carga del dataset** — Lectura desde archivo local
3. **Limpieza inicial** — Manejo de nulos y definición del target
4. **Ingeniería de variables** — Creación de las 12 features expertas
5. **Preparación del split** — División train/test estratificada
6. **Features dependientes del target** — Cálculo SIN data leakage
7. **Entrenamiento del modelo** — XGBoost con early stopping
8. **Evaluación completa** — Métricas, matriz de confusión, ROC/PR curves
9. **Validación cruzada** — Robustez del modelo
10. **Reporte de impacto económico** — Análisis monetario
11. **Importancia de variables** — Interpretabilidad
12. **Optimización con Optuna** *(opcional)*
13. **Persistencia del modelo** — Guardado para producción
14. **Simulador de predicción** — Inferencia sobre clientes individuales

## 1️⃣ Configuración del Entorno
Instalación de dependencias y carga de librerías. Ejecutar la celda de instalación solo la primera vez (en Colab) o si falta alguna librería.

In [ ]:
# 1.1 — Instalación de dependencias (descomentar si es necesario)
# !pip install -q xgboost scikit-learn pandas numpy matplotlib seaborn optuna joblib

In [ ]:
# 1.2 — Imports principales
# Carga todas las librerías que se usarán a lo largo del notebook.
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score, average_precision_score,
    roc_curve, precision_recall_curve, ConfusionMatrixDisplay
)
import xgboost as xgb

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print(f"✅ Librerías cargadas correctamente")
print(f"   - pandas:  {pd.__version__}")
print(f"   - numpy:   {np.__version__}")
print(f"   - xgboost: {xgb.__version__}")

## 2️⃣ Carga del Dataset desde Archivo Local
El dataset se lee desde una ruta relativa (`./data/`). Esto permite ejecutar el notebook en cualquier entorno (local, Colab, GitHub Codespaces) sin depender de Google Drive.

> **📝 Nota:** Si trabajas en **Google Colab**, sube el archivo CSV a una carpeta `data/` en el mismo directorio del notebook, o usa `files.upload()` de la celda alternativa.

In [ ]:
# 2.1 — Definición de la ruta del dataset
# El repositorio incluye una muestra de 10,000 filas para que el notebook
# sea ejecutable inmediatamente. Para usar el dataset completo de 1M, descárgalo
# (instrucciones en el README) y cambia la ruta de abajo.
#
# Estructura esperada:
#   proyecto/
#   ├── MODELO_ESTRATIFICADO_ENTRENADO.ipynb
#   └── data/
#       ├── ecommerce_estratificado_10K.csv   (incluido en el repo)
#       └── ecommerce_estratificado_1M.csv    (descargar aparte, ver README)

# Cambia esta línea para usar el dataset que prefieras:
RUTA_DATASET = './data/ecommerce_estratificado_10K.csv'      # muestra rápida (recomendada para pruebas)
# RUTA_DATASET = './data/ecommerce_estratificado_1M.csv'     # dataset completo (mejores métricas)

# Verificación de existencia
if os.path.exists(RUTA_DATASET):
    tamano_mb = os.path.getsize(RUTA_DATASET) / (1024 * 1024)
    print(f"✅ Archivo encontrado: {RUTA_DATASET}")
    print(f"   Tamaño: {tamano_mb:.2f} MB")
    if tamano_mb < 10:
        print("   ℹ️  Estás usando la muestra pequeña. Las métricas serán modestas")
        print("       (ROC-AUC ~0.60-0.65). Para resultados óptimos, usa el dataset")
        print("       completo de 1M filas (ROC-AUC esperado ~0.90+).")
else:
    print(f"❌ Archivo NO encontrado en: {RUTA_DATASET}")
    print(f"   Directorio actual: {os.getcwd()}")
    print(f"   Verifica que exista la carpeta 'data/' con el CSV adentro.")

In [ ]:
# 2.2 — [ALTERNATIVA PARA GOOGLE COLAB] Subir archivo manualmente
# Descomenta este bloque SOLO si estás en Colab y prefieres subir el archivo desde tu PC.
#
# from google.colab import files
# uploaded = files.upload()  # Abre un selector de archivos del navegador
# RUTA_DATASET = list(uploaded.keys())[0]
# print(f"✅ Archivo subido: {RUTA_DATASET}")

In [ ]:
# 2.3 — Carga eficiente del dataset
# Se lee en chunks para optimizar memoria en máquinas con RAM limitada (Colab gratuito = 12 GB).
def cargar_dataset(ruta, chunksize=500_000):
    """Carga el CSV en pedazos y los concatena en un solo DataFrame."""
    print(f"⏳ Cargando dataset desde {ruta}...")
    chunks = pd.read_csv(ruta, chunksize=chunksize)
    df = pd.concat(chunks, ignore_index=True)
    print(f"✅ Dataset cargado: {len(df):,} filas × {df.shape[1]} columnas")
    print(f"   Memoria utilizada: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
    return df

df = cargar_dataset(RUTA_DATASET)

# Vista rápida
print("\n📊 Primeras 5 filas:")
df.head()

In [ ]:
# 2.4 — Diagnóstico inicial del dataset
# Muestra tipos de datos, valores nulos y estadísticas descriptivas básicas.
print("=" * 60)
print("INFORMACIÓN GENERAL")
print("=" * 60)
df.info()

print("\n" + "=" * 60)
print("VALORES NULOS POR COLUMNA")
print("=" * 60)
nulos = df.isnull().sum()
print(nulos[nulos > 0] if nulos.sum() > 0 else "Sin valores nulos detectados.")

## 3️⃣ Limpieza Inicial y Definición del Target
Imputación de nulos en columnas categóricas, extracción de la categoría principal y construcción de la variable objetivo `is_purchased`.

In [ ]:
# 3.1 — Limpieza y construcción del target
def limpiar_dataset(df):
    """
    - Imputa nulos en columnas categóricas con 'unknown'.
    - Extrae la categoría principal (primer nivel del category_code).
    - Construye el target binario is_purchased (1 = compra, 0 = otro evento).
    """
    df = df.copy()

    # Imputación de nulos
    df['category_code'] = df['category_code'].fillna('unknown')
    df['brand'] = df['brand'].fillna('unknown')

    # Categoría principal (ej: 'electronics.smartphone' → 'electronics')
    df['main_category'] = df['category_code'].str.split('.').str[0]

    # Target binario
    if 'event_type' in df.columns:
        df['is_purchased'] = (df['event_type'] == 'purchase').astype(int)
    else:
        raise ValueError("Columna 'event_type' no encontrada. Revisa el dataset.")

    return df

df = limpiar_dataset(df)

# Distribución del target
print("📊 Distribución de la clase objetivo (is_purchased):")
dist = df['is_purchased'].value_counts(normalize=True) * 100
print(f"   Clase 0 (no compra): {dist[0]:.2f}%")
print(f"   Clase 1 (compra):    {dist[1]:.2f}%")
print(f"   Ratio de desbalance: {dist[0] / dist[1]:.2f} : 1")

## 4️⃣ Ingeniería de Variables (Parte Segura)
Aquí se generan las features que **NO dependen del target**, por lo que se pueden calcular sobre todo el dataset sin riesgo de *data leakage* (filtración de información del test set hacia el entrenamiento).

**Variables creadas en este bloque:**
- Temporales: `hour`, `day_of_week`, `is_weekend`
- Codificadas: `brand_n`, `cat_n`
- Económicas: `price_ratio_cat`, `price_vs_brand`
- Popularidad estática: `brand_popularity`

In [ ]:
# 4.1 — Generar variables que no dependen del target
def generar_features_seguras(df):
    """Crea features que no usan información del target (is_purchased)."""
    df = df.copy()
    print("⏳ Generando features temporales y económicas...")

    # --- Temporales ---
    df['event_time'] = pd.to_datetime(df['event_time']).dt.tz_localize(None)
    df['hour'] = df['event_time'].dt.hour.astype('int8')
    df['day_of_week'] = df['event_time'].dt.dayofweek.astype('int8')
    df['is_weekend'] = df['day_of_week'].isin([5, 6]).astype('int8')

    # --- Codificación de categóricas (Label Encoding) ---
    le_brand = LabelEncoder()
    le_cat = LabelEncoder()
    df['brand_n'] = le_brand.fit_transform(df['brand'].astype(str))
    df['cat_n'] = le_cat.fit_transform(df['main_category'].astype(str))

    # --- Variables de contexto económico ---
    avg_price_cat = df.groupby('cat_n')['price'].transform('mean')
    df['price_ratio_cat'] = df['price'] / avg_price_cat

    avg_price_brand = df.groupby('brand_n')['price'].transform('mean')
    df['price_vs_brand'] = df['price'] / avg_price_brand

    # --- Popularidad estática (cuenta de eventos, no usa target) ---
    df['brand_popularity'] = df['brand'].map(df['brand'].value_counts()).astype('int32')

    print("✅ Features seguras generadas.")
    return df, le_brand, le_cat

df, le_brand, le_cat = generar_features_seguras(df)
print(f"\n   Total de columnas ahora: {df.shape[1]}")

## 5️⃣ División Train/Test Estratificada
Antes de calcular features dependientes del target, se separa el dataset. El parámetro `stratify=y` garantiza que la proporción de compras se mantenga igual en train y test, lo cual es crítico en datasets desbalanceados.

In [ ]:
# 5.1 — Selección preliminar de columnas y split estratificado
features_independientes = [
    'price', 'brand_n', 'cat_n', 'hour', 'day_of_week', 'is_weekend',
    'price_ratio_cat', 'price_vs_brand', 'brand_popularity'
]

# Necesitamos también las columnas auxiliares para calcular features dependientes después
columnas_auxiliares = ['product_id', 'user_session', 'is_purchased']

# Split inicial (sobre el dataframe completo para mantener las columnas auxiliares)
df_train, df_test = train_test_split(
    df[features_independientes + columnas_auxiliares],
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=df['is_purchased']
)

print(f"✅ Split estratificado completado:")
print(f"   Train: {len(df_train):,} filas ({len(df_train)/len(df)*100:.1f}%)")
print(f"   Test:  {len(df_test):,} filas ({len(df_test)/len(df)*100:.1f}%)")
print(f"\n   % compras en train: {df_train['is_purchased'].mean()*100:.3f}%")
print(f"   % compras en test:  {df_test['is_purchased'].mean()*100:.3f}%")

## 6️⃣ Features Dependientes del Target — SIN Data Leakage
Esta es **la mejora más importante del notebook**. Variables como `cat_conv_rate` (tasa de conversión por categoría) y `product_pop` (popularidad del producto) se calculan **únicamente sobre el conjunto de entrenamiento** y luego se mapean al test.

**¿Por qué importa?** Si las calculáramos sobre todo el dataset, el modelo estaría "viendo" información del futuro (test) durante el entrenamiento, inflando artificialmente sus métricas.

In [ ]:
# 6.1 — Calcular features dependientes del target SOLO sobre train
def agregar_features_dependientes(df_train, df_test):
    """
    Calcula features que usan is_purchased (o agregaciones sensibles al target)
    únicamente sobre train, y las mapea al test usando los mismos diccionarios.
    Esto previene data leakage.
    """
    print("⏳ Calculando features dependientes del target sobre TRAIN...")

    # --- Tasa de conversión por categoría (usa is_purchased → solo train) ---
    cat_conv_map = df_train.groupby('cat_n')['is_purchased'].mean()
    df_train['cat_conv_rate'] = df_train['cat_n'].map(cat_conv_map)
    df_test['cat_conv_rate'] = df_test['cat_n'].map(cat_conv_map).fillna(cat_conv_map.mean())

    # --- Popularidad de producto (cuenta de eventos en train) ---
    prod_pop_map = df_train.groupby('product_id').size()
    df_train['product_pop'] = df_train['product_id'].map(prod_pop_map).astype('int32')
    df_test['product_pop'] = df_test['product_id'].map(prod_pop_map).fillna(0).astype('int32')

    # --- Intensidad de sesión (eventos por sesión en train) ---
    sess_int_map = df_train.groupby('user_session').size()
    df_train['session_intensity'] = df_train['user_session'].map(sess_int_map).astype('int32')
    df_test['session_intensity'] = df_test['user_session'].map(sess_int_map).fillna(1).astype('int32')

    print("✅ Features dependientes calculadas sin leakage.")
    return df_train, df_test

df_train, df_test = agregar_features_dependientes(df_train, df_test)

# Verificación
print("\n📊 Estadísticas de las nuevas features en train:")
print(df_train[['cat_conv_rate', 'product_pop', 'session_intensity']].describe().round(3))

In [ ]:
# 6.2 — Construir matrices X, y finales
features_finales = [
    'price', 'brand_n', 'cat_n', 'hour', 'day_of_week', 'is_weekend',
    'price_ratio_cat', 'price_vs_brand', 'brand_popularity',
    'product_pop', 'session_intensity', 'cat_conv_rate'
]

X_train = df_train[features_finales].fillna(0)
y_train = df_train['is_purchased']
X_test = df_test[features_finales].fillna(0)
y_test = df_test['is_purchased']

print(f"✅ Matrices listas para entrenamiento:")
print(f"   X_train: {X_train.shape}  |  y_train: {y_train.shape}")
print(f"   X_test:  {X_test.shape}  |  y_test:  {y_test.shape}")
print(f"\n   Total de features: {len(features_finales)}")

## 7️⃣ Entrenamiento del Modelo XGBoost
Configuración con hiperparámetros optimizados y **early stopping**: el entrenamiento se detiene automáticamente si el modelo deja de mejorar, ahorrando tiempo y previniendo overfitting.

In [ ]:
# 7.1 — Cálculo del peso de balanceo
# Como hay muchas más visitas que compras, ajustamos el peso de la clase minoritaria.
ratio_desbalance = float(y_train.value_counts()[0]) / y_train.value_counts()[1]
print(f"📊 Ratio de desbalance (clase 0 / clase 1): {ratio_desbalance:.2f}")
print(f"   scale_pos_weight a usar: {ratio_desbalance * 0.9:.2f}")

In [ ]:
# 7.2 — Configuración y entrenamiento del modelo maestro
params = {
    'n_estimators': 500,             # Más árboles, pero con early stopping
    'max_depth': 8,                  # Profundidad para relaciones complejas
    'learning_rate': 0.0571,         # Tasa de aprendizaje optimizada
    'subsample': 0.8,                # 80% de filas por árbol
    'colsample_bytree': 0.8,         # 80% de features por árbol
    'scale_pos_weight': ratio_desbalance * 0.9,
    'eval_metric': ['logloss', 'auc'],
    'tree_method': 'hist',           # Método rápido (recomendado para datasets grandes)
    'random_state': RANDOM_STATE,
    'early_stopping_rounds': 25,     # Detener si no mejora en 25 rondas
    'n_jobs': -1
}

print("⏳ Entrenando modelo maestro... (puede tomar 2-4 minutos)")
modelo_maestro = xgb.XGBClassifier(**params)
modelo_maestro.fit(
    X_train, y_train,
    eval_set=[(X_train, y_train), (X_test, y_test)],
    verbose=50  # Imprime progreso cada 50 árboles
)

print(f"\n✅ Entrenamiento completado.")
print(f"   Mejor iteración: {modelo_maestro.best_iteration}")
print(f"   Mejor score (AUC): {modelo_maestro.best_score:.4f}")

## 8️⃣ Evaluación Completa del Modelo
Métricas de clasificación (precision, recall, F1), ROC-AUC, PR-AUC y matriz de confusión visual.

> **¿Por qué tantas métricas?** En datasets desbalanceados, *accuracy* engaña. Un modelo que predice "no compra" siempre tendría ~96% de accuracy y sería inútil. **PR-AUC** y **Recall** son las métricas que realmente importan aquí.

In [ ]:
# 8.1 — Predicciones y métricas básicas
y_pred = modelo_maestro.predict(X_test)
y_proba = modelo_maestro.predict_proba(X_test)[:, 1]

print("=" * 60)
print("REPORTE DE CLASIFICACIÓN")
print("=" * 60)
print(classification_report(y_test, y_pred, target_names=['No compra', 'Compra']))

print("=" * 60)
print("MÉTRICAS PROBABILÍSTICAS")
print("=" * 60)
roc_auc = roc_auc_score(y_test, y_proba)
pr_auc = average_precision_score(y_test, y_proba)
print(f"ROC-AUC: {roc_auc:.4f}  (perfecto = 1.0, aleatorio = 0.5)")
print(f"PR-AUC:  {pr_auc:.4f}  (más informativa para clases desbalanceadas)")

In [ ]:
# 8.2 — Visualización: matriz de confusión + curvas ROC y PR
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# --- Matriz de confusión ---
cm = confusion_matrix(y_test, y_pred)
ConfusionMatrixDisplay(cm, display_labels=['No compra', 'Compra']).plot(
    ax=axes[0], cmap='Blues', values_format=',d', colorbar=False
)
axes[0].set_title('Matriz de Confusión', fontsize=13, fontweight='bold')
axes[0].grid(False)

# --- Curva ROC ---
fpr, tpr, _ = roc_curve(y_test, y_proba)
axes[1].plot(fpr, tpr, color='#2E86AB', lw=2, label=f'ROC (AUC = {roc_auc:.3f})')
axes[1].plot([0, 1], [0, 1], 'k--', lw=1, alpha=0.5)
axes[1].set_xlabel('Tasa de Falsos Positivos')
axes[1].set_ylabel('Tasa de Verdaderos Positivos')
axes[1].set_title('Curva ROC', fontsize=13, fontweight='bold')
axes[1].legend(loc='lower right')

# --- Curva Precision-Recall ---
precision, recall, _ = precision_recall_curve(y_test, y_proba)
axes[2].plot(recall, precision, color='#A23B72', lw=2, label=f'PR (AUC = {pr_auc:.3f})')
baseline = y_test.mean()
axes[2].axhline(baseline, color='k', linestyle='--', lw=1, alpha=0.5,
                label=f'Baseline ({baseline:.3f})')
axes[2].set_xlabel('Recall')
axes[2].set_ylabel('Precision')
axes[2].set_title('Curva Precision-Recall', fontsize=13, fontweight='bold')
axes[2].legend(loc='lower left')

plt.tight_layout()
plt.show()

## 9️⃣ Validación Cruzada Estratificada
Un único split puede ser engañoso por azar. Con 5-fold CV evaluamos el modelo en 5 particiones distintas y reportamos media ± desviación estándar. Si la varianza es baja, el modelo es robusto.

In [ ]:
# 9.1 — Cross-validation con StratifiedKFold (5 folds)
# Usamos un modelo simplificado (sin early stopping) porque CV requiere fit completo en cada fold.
print("⏳ Ejecutando validación cruzada (5-fold)... esto toma ~3-5 min")

modelo_cv = xgb.XGBClassifier(
    n_estimators=modelo_maestro.best_iteration or 200,
    max_depth=8,
    learning_rate=0.0571,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=ratio_desbalance * 0.9,
    tree_method='hist',
    random_state=RANDOM_STATE,
    n_jobs=-1
)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
scores_auc = cross_val_score(modelo_cv, X_train, y_train, cv=skf, scoring='roc_auc', n_jobs=-1)
scores_pr  = cross_val_score(modelo_cv, X_train, y_train, cv=skf, scoring='average_precision', n_jobs=-1)

print("\n✅ Resultados de validación cruzada:")
print(f"   ROC-AUC: {scores_auc.mean():.4f} ± {scores_auc.std():.4f}")
print(f"   PR-AUC:  {scores_pr.mean():.4f} ± {scores_pr.std():.4f}")
print(f"\n   Folds individuales (ROC-AUC): {[f'{s:.4f}' for s in scores_auc]}")

## 🔟 Reporte de Impacto Económico
Traducimos las métricas técnicas a lenguaje de negocio: cuánto dinero captura el modelo, cuánto pierde y qué tan eficiente es la captura del flujo de efectivo.

In [ ]:
# 10.1 — Reporte financiero ejecutivo
def reporte_impacto_economico(X_test, y_test, modelo):
    """Calcula el impacto monetario de las predicciones del modelo."""
    analisis = pd.DataFrame({
        'precio': X_test['price'].values,
        'realidad': y_test.values,
        'prediccion': modelo.predict(X_test)
    })

    ventas_reales = analisis.loc[analisis['realidad'] == 1, 'precio'].sum()
    ventas_capturadas = analisis.loc[(analisis['realidad'] == 1) & (analisis['prediccion'] == 1), 'precio'].sum()
    ventas_perdidas = analisis.loc[(analisis['realidad'] == 1) & (analisis['prediccion'] == 0), 'precio'].sum()
    dinero_riesgo = analisis.loc[(analisis['realidad'] == 0) & (analisis['prediccion'] == 1), 'precio'].sum()

    eficiencia = (ventas_capturadas / ventas_reales) * 100
    pct_clientes_objetivo = (analisis['prediccion'].mean()) * 100

    print("=" * 60)
    print("   📈 INFORME DE IMPACTO ECONÓMICO")
    print("=" * 60)
    print(f"  Ventas reales totales en el set: ${ventas_reales:>15,.2f}")
    print(f"  Ventas capturadas por la IA:     ${ventas_capturadas:>15,.2f}")
    print(f"  Ventas no detectadas:            ${ventas_perdidas:>15,.2f}")
    print("-" * 60)
    print(f"  EFICIENCIA DE CAPTURA (Recall$): {eficiencia:>14.2f}%")
    print(f"  Dinero en riesgo (FP):           ${dinero_riesgo:>15,.2f}")
    print(f"  % clientes objetivo de marketing: {pct_clientes_objetivo:>13.2f}%")
    print("=" * 60)
    print("\n💡 INTERPRETACIÓN ESTRATÉGICA:")
    print(f"   • El modelo asegura el {eficiencia:.1f}% del flujo de efectivo real.")
    print(f"   • Permite enfocar marketing solo al {pct_clientes_objetivo:.1f}% de los clientes")
    print(f"     con mayor probabilidad de compra, optimizando el presupuesto.")

reporte_impacto_economico(X_test, y_test, modelo_maestro)

## 1️⃣1️⃣ Importancia de Variables
¿Qué features están aportando más al modelo? Esto es clave para la interpretabilidad y para defender el modelo en una presentación.

In [ ]:
# 11.1 — Gráfico de importancia (gain = ganancia promedio aportada por cada feature)
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

xgb.plot_importance(modelo_maestro, ax=axes[0], importance_type='gain',
                    max_num_features=12, height=0.6,
                    title='Importancia por GANANCIA (gain)',
                    xlabel='Ganancia promedio', ylabel='Variables', color='#2E86AB')

xgb.plot_importance(modelo_maestro, ax=axes[1], importance_type='weight',
                    max_num_features=12, height=0.6,
                    title='Importancia por FRECUENCIA (weight)',
                    xlabel='Veces usada en splits', ylabel='', color='#A23B72')

for ax in axes:
    ax.grid(True, linestyle='--', alpha=0.4)

plt.tight_layout()
plt.show()

# Tabla numérica con ambas métricas
importancias = pd.DataFrame({
    'feature': features_finales,
    'gain': modelo_maestro.feature_importances_,
}).sort_values('gain', ascending=False)
print("\n📊 Ranking de variables por GAIN:")
print(importancias.to_string(index=False))

## 1️⃣2️⃣ Optimización de Hiperparámetros con Optuna *(Opcional)*
**⚠️ Esta celda es opcional** — toma ~10-15 minutos extra. Optuna prueba combinaciones inteligentes de hiperparámetros y devuelve la mejor configuración encontrada.

Útil si quieres squeeze el último 1-2% de performance o si necesitas mostrar tuning sistemático en tu defensa.

In [ ]:
# 12.1 — Búsqueda bayesiana con Optuna (DESCOMENTAR PARA EJECUTAR)
# import optuna
# from sklearn.model_selection import cross_val_score
#
# def objective(trial):
#     params = {
#         'n_estimators': trial.suggest_int('n_estimators', 200, 600),
#         'max_depth': trial.suggest_int('max_depth', 4, 12),
#         'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
#         'subsample': trial.suggest_float('subsample', 0.6, 1.0),
#         'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
#         'scale_pos_weight': ratio_desbalance * trial.suggest_float('spw_mult', 0.5, 1.2),
#         'tree_method': 'hist',
#         'random_state': RANDOM_STATE,
#         'n_jobs': -1
#     }
#     modelo = xgb.XGBClassifier(**params)
#     # Submuestreo para acelerar (usa 30% del train en cada trial)
#     X_sub = X_train.sample(frac=0.3, random_state=RANDOM_STATE)
#     y_sub = y_train.loc[X_sub.index]
#     score = cross_val_score(modelo, X_sub, y_sub, cv=3, scoring='average_precision', n_jobs=-1).mean()
#     return score
#
# study = optuna.create_study(direction='maximize',
#                              sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE))
# study.optimize(objective, n_trials=20, show_progress_bar=True)
#
# print(f"\n✅ Mejores parámetros encontrados:")
# for k, v in study.best_params.items():
#     print(f"   {k}: {v}")
# print(f"\n   Mejor PR-AUC: {study.best_value:.4f}")
print("ℹ️  Celda comentada por defecto. Descomenta el código para ejecutar Optuna.")

## 1️⃣3️⃣ Persistencia del Modelo
Guardamos el modelo entrenado y los `LabelEncoders` para poder usarlos en producción sin reentrenar.

In [ ]:
# 13.1 — Guardar modelo y encoders
os.makedirs('./models', exist_ok=True)

# Modelo XGBoost en formato nativo (más eficiente que pickle)
modelo_maestro.save_model('./models/modelo_maestro.json')

# Encoders y metadata con joblib
joblib.dump(le_brand, './models/le_brand.pkl')
joblib.dump(le_cat, './models/le_cat.pkl')
joblib.dump({
    'features': features_finales,
    'ratio_desbalance': ratio_desbalance,
    'best_iteration': modelo_maestro.best_iteration,
    'roc_auc': roc_auc,
    'pr_auc': pr_auc
}, './models/metadata.pkl')

print("✅ Modelo guardado en ./models/")
print("   - modelo_maestro.json")
print("   - le_brand.pkl, le_cat.pkl")
print("   - metadata.pkl")

## 1️⃣4️⃣ Simulador de Predicción
Función para predecir la probabilidad de compra de un cliente específico dado un perfil (precio, marca, categoría, hora, fin de semana).

In [ ]:
# 14.1 — Función de simulación de cliente individual
def simular_intencion_compra(precio, marca, categoria, hora, es_fin_de_semana,
                             modelo=modelo_maestro, df_ref=df_train):
    """
    Predice la probabilidad de compra de un cliente con el perfil dado.

    Parámetros:
        precio: precio del producto en USD
        marca: nombre de la marca (ej: 'apple')
        categoria: categoría principal (ej: 'electronics')
        hora: hora del día (0-23)
        es_fin_de_semana: 1 si es sábado/domingo, 0 si no
    """
    print(f"--- 🔍 ANALIZANDO PERFIL DE CLIENTE ---")

    # Encoding de categóricas (con fallback a 'unknown')
    try:
        brand_n = le_brand.transform([marca])[0]
    except ValueError:
        brand_n = le_brand.transform(['unknown'])[0]
    try:
        cat_n = le_cat.transform([categoria])[0]
    except ValueError:
        cat_n = le_cat.transform(['unknown'])[0]

    # Estimaciones basadas en el train set (NO en todo el dataset, para coherencia)
    brand_pop = df_ref.loc[df_ref['brand_n'] == brand_n, 'brand_popularity']
    brand_pop = brand_pop.iloc[0] if len(brand_pop) > 0 else 0

    avg_cat_price = df_ref.loc[df_ref['cat_n'] == cat_n, 'price'].mean()
    price_ratio_cat = precio / avg_cat_price if avg_cat_price > 0 else 1.0

    avg_brand_price = df_ref.loc[df_ref['brand_n'] == brand_n, 'price'].mean()
    price_vs_brand = precio / avg_brand_price if avg_brand_price > 0 else 1.0

    prod_pop = df_ref.loc[df_ref['cat_n'] == cat_n, 'product_pop'].median()
    session_intensity = df_ref['session_intensity'].median()
    cat_conv_rate = df_ref.loc[df_ref['cat_n'] == cat_n, 'cat_conv_rate'].mean()

    # Vector de entrada (orden EXACTO de features_finales)
    vector = pd.DataFrame([[
        precio, brand_n, cat_n, hora, 0, es_fin_de_semana,
        price_ratio_cat, price_vs_brand, brand_pop,
        prod_pop, session_intensity, cat_conv_rate
    ]], columns=features_finales)

    probabilidad = modelo.predict_proba(vector)[0][1]

    print(f"  Producto: {categoria} | Marca: {marca} | Precio: ${precio:.2f}")
    print(f"  Hora: {hora}h | Fin de semana: {'Sí' if es_fin_de_semana else 'No'}")
    print(f"\n  🎯 PROBABILIDAD DE COMPRA: {probabilidad:.2%}")

    if probabilidad > 0.5:
        print("  💰 RESULTADO: ALTA probabilidad de venta → activar campaña de marketing.")
    elif probabilidad > 0.25:
        print("  ⚖️  RESULTADO: Probabilidad MEDIA → considerar retargeting.")
    else:
        print("  👀 RESULTADO: Probabilidad BAJA → cliente probablemente solo navegando.")

    return probabilidad

# --- PRUEBAS DEL SIMULADOR ---
print("=" * 60)
_ = simular_intencion_compra(precio=800, marca='apple', categoria='electronics',
                              hora=15, es_fin_de_semana=1)
print("\n" + "=" * 60)
_ = simular_intencion_compra(precio=50, marca='samsung', categoria='electronics',
                              hora=10, es_fin_de_semana=0)
print("\n" + "=" * 60)
_ = simular_intencion_compra(precio=1200, marca='unknown', categoria='appliances',
                              hora=22, es_fin_de_semana=1)

---
## ✅ Notebook Completado

**Próximos pasos sugeridos:**
- Ejecutar la celda de Optuna si quieres mejorar las métricas un 1-2% adicional.
- Probar el simulador con perfiles personalizados.
- Cargar el modelo guardado (`./models/modelo_maestro.json`) en otro notebook para inferencia.

**Mejoras técnicas implementadas vs. versión original:**
1. ✅ Eliminación de data leakage en `cat_conv_rate`, `product_pop`, `session_intensity`
2. ✅ Split estratificado para preservar balance de clases
3. ✅ Early stopping para evitar overfitting
4. ✅ Validación cruzada estratificada (5-fold)
5. ✅ Métricas adicionales: ROC-AUC, PR-AUC, matriz visual
6. ✅ Optimización de tipos de datos (int8/int32) → menor uso de memoria
7. ✅ Persistencia del modelo y encoders
8. ✅ Optuna opcional para tuning de hiperparámetros
9. ✅ Lectura desde archivo local (compatible con GitHub)